# Bước 4: Chấm điểm Cảm xúc và Điểm Tương đối (Sentiment Analysis and Comparative Scoring)

Jupyter Notebook này thực hiện **Phần 2.2** theo đề cương phương pháp tính Tỷ trọng Greenwashing (Greenwashing Severity Index - GSI).

### Quy trình dựa trên Phương pháp:
1. **Cơ chế gán trọng số (Weighting Mechanism):** Load file `ESG.xlsx` với các thuật ngữ đã được chuyên gia gán điểm (từ -3 đến +3).
2. **Đồng Bộ Hoá Từ Điển:** Đưa các từ trong từ điển đi qua thuật toán Word Segmentation (`pyvi`) giống như bước `gold` ở đợt trước để khớp định dạng (ví dụ ghép câu `thân_thiện`).
3. **Tính Total Score (Aggregation):** Duyệt độc lập từng bản ghi (Tin tức hoặc Báo cáo). Khớp chuỗi chính xác để đếm số lần sử dụng ngôn từ "Tẩy Xanh"/"Thực Tế" rồi nhân với Trọng số.
4. **Chuẩn hoá (Data Normalization):** Chia `Total Score` cho `Total Number of Tokens` để có `Comparative Score`.
5. **Kết quả cuối:** Lưu dữ liệu đầy đủ (kể cả chi tiết các từ tìm thấy) vào mục `scoring/` (Thành phẩm cấp dữ liệu mới).

In [6]:
import pandas as pd

import json
import re
from pathlib import Path
from pyvi import ViTokenizer

# =============== CẤU HÌNH ĐƯỜNG DẪN =============== 
# --- CẤU HÌNH NĂM VÀ NGÂN HÀNG CẦN XỬ LÝ ---
TARGET_YEAR = "2023"
TARGET_BANK = "Vietcombank"  # Đổi thành "vietcombank", "bidv", v.v. hoặc "*" nếu muốn chạy tất cả ngân hàng trong năm
# ------------------------------------------

# GÁN THỦ CÔNG ĐƯỜNG DẪN TRỰC TIẾP TỚI FILE PDF BÁO CÁO CỦA NGÂN HÀNG
# Bạn tự điền chính xác đường dẫn file PDF vào biến này. (Nhớ có chữ r ở trước ngoặc kép để tránh lỗi đường dẫn Windows)
MANUAL_PDF_PATH = r"D://NCKH//ESG_reporting//Vietcombank_2023_ESG.pdf"  # <-- ĐIỀN ĐƯỜNG DẪN FILE PDF CỦA BẠN VÀO ĐÂY


PROJECT_ROOT = Path.cwd() 
# Trỏ DATA_DIR tới đúng thư mục năm tương ứng
DATA_DIR = PROJECT_ROOT / "data" / TARGET_YEAR

# File từ điển nằm ở ngoài folder Thread_1
ESG_DICT_PATH = PROJECT_ROOT.parent / "ESG_Dictionary" / "ESG.xlsx"

print(f"Năm đang xử lý: {TARGET_YEAR}")
print(f"Ngân hàng đang xử lý: {TARGET_BANK}")
print(f"Thư mục hiện tại: {PROJECT_ROOT}")
print(f"Thư mục Data: {DATA_DIR}")
print(f"Đường dẫn file Từ điển ESG: {ESG_DICT_PATH}")

Năm đang xử lý: 2023
Ngân hàng đang xử lý: Vietcombank
Thư mục hiện tại: d:\NCKH\Thread_1
Thư mục Data: d:\NCKH\Thread_1\data\2023
Đường dẫn file Từ điển ESG: d:\NCKH\ESG_Dictionary\ESG.xlsx


In [7]:
# =============== BƯỚC 1: LOAD VÀ PREPROCESS TỪ ĐIỂN =============== 
df_esg = pd.read_excel(ESG_DICT_PATH)
df_esg.columns = ['Greenwashing_Term', 'Weight']

def preprocess_dict_term(term):
    if pd.isna(term):
        return ""
    # Tiền xử lý giống file Gold: Chữ thường, bỏ dấu câu
    term_cleaned = re.sub(r'[^\w\s]', ' ', str(term).lower())
    # Chạy qua Word Segmentation y hệt để tạo dấu nối các từ ghép
    return ViTokenizer.tokenize(term_cleaned.strip())

# Cập nhật từ điển bằng các từ đã Lemmatize/Tokenize theo định dạng Pyvi
df_esg['Processed_Term'] = df_esg['Greenwashing_Term'].apply(preprocess_dict_term)

# Lưu thành Dictionary tra cứu để tối ưu
# Format: {'thân_thiện với môi_trường': -1, 'xanh': -1, 'tái_tạo': 3, ...}
esg_dict = dict(zip(df_esg['Processed_Term'], df_esg['Weight']))
esg_dict = {k: v for k, v in esg_dict.items() if k} # Bỏ các chuỗi rỗng

print(f"Đã tải {len(df_esg)} từ vựng.")
print("Hiển thị bộ từ điển đã Preprocess (10 mẫu đầu):")
display(df_esg.head(10))

Đã tải 821 từ vựng.
Hiển thị bộ từ điển đã Preprocess (10 mẫu đầu):


,Greenwashing_Term,Weight,Processed_Term
0,thân thiện với môi trường,-1,thân_thiện với môi_trường
1,xanh,-1,xanh
2,tái tạo,3,tái_tạo
3,không thể tái tạo,-3,không_thể tái_tạo
4,ít tác động,-1,ít tác_động
5,tác động thấp,-1,tác_động thấp
6,có trách nhiệm,1,có trách_nhiệm
7,bền vững,1,bền_vững
8,đạo đức,1,đạo_đức
9,phi đạo đức,-2,phi đạo_đức


In [8]:
# =============== BƯỚC 2: TRAVERSE TEXT VÀ TÍNH ĐIỂM TỔNG CỘNG =============== 

def calculate_scores(text):
    """
    Hàm khớp từ điển, đếm số lượng Tokens, trả về Total Score, 
    Comparative Score, và Danh sách các từ trùng khớp.
    """
    if not text or not isinstance(text, str):
        return {
            "total_tokens": 0,
            "total_score": 0,
            "comparative_score": 0,
            "esg_matched_terms": {}
        }

    # Tổng số Tokens bằng cách chia tách qua khoảng trắng (space)
    total_tokens = len(text.split())
    total_score = 0
    matched_terms = {}

    # Bọc khoảng trắng để Regex / Đếm không bị dính chữ (vd: ' xanh ' không dính 'cây_xanh')
    padded_text = f" {text} "
    
    for term, weight in esg_dict.items():
        # Tương tự, bọc Word Segmented Term trong khoảng trắng để exact match
        padded_term = f" {term} "
        
        # Đếm tần suất cụm từ này xuất hiện trong đoạn
        freq_count = padded_text.count(padded_term)
        
        if freq_count > 0:
            subtotal = freq_count * weight
            total_score += subtotal
            matched_terms[term] = {
                "frequency": freq_count,
                "weight": weight,
                "subtotal": subtotal
            }
            
    # Data Normalization (Chuẩn hóa Comparative Score)
    #   Comparative Score = Total Score / Total number of Tokens
    comparative_score = total_score / total_tokens if total_tokens > 0 else 0
    
    return {
        "total_tokens": total_tokens,
        "total_score": total_score,
        "comparative_score": comparative_score,
        "esg_matched_terms": matched_terms
    }

In [9]:
# =============== BƯỚC 3: QUÉT VÀ CHẤM ĐIỂM FOLDER GOLD =============== 

bank_path = "**" if TARGET_BANK == "*" else TARGET_BANK
gold_files = list(DATA_DIR.glob(f"{bank_path}/gold/*.json"))
print(f"Tìm thấy {len(gold_files)} file tập dữ liệu ở tầng Gold của ngân hàng {TARGET_BANK}.")

for file_path in gold_files:
    print(f"\nĐang xử lý file: {file_path.name}")
    
    with open(file_path, 'r', encoding='utf-8') as f:
        json_data = json.load(f)
        
    # JSON output từ bước gold là dạng dict { "bank_name": [danh sách các items] }
    # Thay vì .get("items"), thực hiện traverse linh hoạt theo cấu trúc thực tế
    if isinstance(json_data, dict):
        bank_or_items = json_data
        scored_records_dict = {}
        for b_name, items in bank_or_items.items():
            if not isinstance(items, list):
                continue
            scored_tmp = []
            for item in items:
                preprocessed_text = item.get("preprocessed_content", "")
                score_results = calculate_scores(preprocessed_text)
                
                new_item = item.copy()
                new_item.update(score_results)
                scored_tmp.append(new_item)
            scored_records_dict[b_name] = scored_tmp
        output_data = scored_records_dict
        total_items_scored = sum(len(ls) for ls in scored_records_dict.values())
    else:  # Dự phòng nếu data là list thuần
        items = json_data if isinstance(json_data, list) else []
        scored_records = []
        for item in items:
            preprocessed_text = item.get("preprocessed_content", "")
            score_results = calculate_scores(preprocessed_text)
            
            new_item = item.copy()
            new_item.update(score_results)
            scored_records.append(new_item)
        output_data = scored_records
        total_items_scored = len(scored_records)
        
    # =============== TẠO FOLDER SCORING VÀ SAVE KẾT QUẢ =============== 
    # Lấy thư mục bank (parent của gold/) sau đó tạo mục 'scoring'
    scoring_dir = file_path.parent.parent / "scoring"
    scoring_dir.mkdir(parents=True, exist_ok=True)
    
    # Thay vì đè tên cũ, dán nhãn 'scored_'
    out_filename = f"scored_{file_path.name.replace('preprocessed_', '')}"
    output_path = scoring_dir / out_filename
    
    with open(output_path, 'w', encoding='utf-8') as out_f:
        json.dump(output_data, out_f, ensure_ascii=False, indent=4)
        
    print(f"✓ Đã chấm điểm xong và lưu {total_items_scored} Article/Report tại: {output_path}")

print("\n🚀 HOÀN TẤT GIAI ĐOẠN NORMALIZATION VÀ SCORING (THÀNH CÔNG)!")

Tìm thấy 1 file tập dữ liệu ở tầng Gold của ngân hàng Vietcombank.

Đang xử lý file: preprocessed_esg_check_llms_20260404_205927.json
✓ Đã chấm điểm xong và lưu 35 Article/Report tại: d:\NCKH\Thread_1\data\2023\Vietcombank\scoring\scored_esg_check_llms_20260404_205927.json

🚀 HOÀN TẤT GIAI ĐOẠN NORMALIZATION VÀ SCORING (THÀNH CÔNG)!


# =============== BƯỚC 4: CHẤM ĐIỂM CHUYÊN SÂU CHO BÁO CÁO (REPORTS) =============== 
Đoạn code dưới đây hỗ trợ bạn trích xuất chữ từ tệp Báo cáo công ty (định dạng PDF) lưu trong folder `ESG_reporting`.
Quá trình xử lý:
1. Đọc nội dung file PDF bằng thư viện `pdfplumber` (giữ nguyên bố cục được tốt hơn).
2. Lưu nội dung thô vừa chuyển đổi ra một file `.txt` để bạn tiện kiểm tra lại (đúng như yêu cầu).
3. Tiền xử lý văn bản báo cáo (chữ thường, xóa dấu câu, và phân mảnh từ bằng `pyvi` tương tự thuật toán ở bước "Gold").
4. Đưa văn bản vào hàm `calculate_scores()` để tính Điểm tương đối (Comparative Score) nhằm có cơ sở so sánh với tin tức cùng công ty.

In [11]:
# Cài đặt thư viện xử lý PDF (Hãy uncomment và chạy 1 lần nếu gặp lỗi thiếu thư viện)
# !pip install pdfplumber

import re
import json


pdf_target = Path(MANUAL_PDF_PATH)

if not pdf_target.exists() or not pdf_target.is_file():
    print(f"❌ LỖI: Không tìm thấy file hoặc sai đường dẫn: {MANUAL_PDF_PATH}")
    print("Vui lòng check lại biến MANUAL_PDF_PATH!")
    report_files = []
else:
    report_files = [pdf_target]
    print(f"Đã tải thành công file báo cáo: {pdf_target.name}")

# Thư mục con để lưu file text vừa extract (trích xuất) từ PDF (Lưu tại cùng folder chứa file PDF)
if pdf_target.exists():
    REPORT_TEXT_DIR = pdf_target.parent / "extracted_texts"
    REPORT_TEXT_DIR.mkdir(parents=True, exist_ok=True)

# Tương tự như xử lý tin tức, văn bản cần phải lower-case, xóa dấu phân cách và Word Segmentation
def preprocess_report_text(text):
    if not text:
        return ""
    text_cleaned = re.sub(r'[^\w\s]', ' ', str(text).lower())
    return ViTokenizer.tokenize(text_cleaned.strip())

report_scores = []

for pdf_path in report_files:
    print(f"\nĐang xử lý báo cáo: {pdf_path.name}")
    full_text = ""
    
    # 1. TRÍCH XUẤT TEXT TỪ PDF (Ưu tiên dùng pdfplumber)
    try:
        import pdfplumber
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    full_text += page_text + "\n"
    except ImportError:
        print("LỖI: Vui lòng cài đặt pdfplumber bằng lệnh `!pip install pdfplumber` để đọc file pdf.")
        break
    except Exception as e:
        print(f"Lỗi khi đọc file pdf {pdf_path.name}: {e}")
        continue
        
    if not full_text.strip():
        print(f"  -> File {pdf_path.name} không trích xuất được văn bản.")
        continue
        
    # 2. LƯU TEXT THÔ THEO YÊU CẦU 
    txt_filename = pdf_path.stem + "_raw.txt"
    txt_filepath = REPORT_TEXT_DIR / txt_filename
    with open(txt_filepath, 'w', encoding='utf-8') as f:
        f.write(full_text)
    print(f"  -> Đã trích xuất và lưu nội dung text thô vào: {txt_filepath.name}")
        
    # 3. TIỀN XỬ LÝ (Word Segmentation với pyvi)
    print("  -> Đang tiền xử lý (Word Segmentation) văn bản...")
    preprocessed_text = preprocess_report_text(full_text)
    
    # 4. CHẤM ĐIỂM BẰNG HÀM CŨ
    print("  -> Đang tính toán điểm Total Score và Comparative Score...")
    score_results = calculate_scores(preprocessed_text)
    
    # Nối dữ liệu
    report_data = {
        "report_name": pdf_path.name,
        "text_file_path": str(txt_filepath),
        **score_results
    }
    report_scores.append(report_data)

# =============== LƯU KẾT QUẢ ĐIỂM BÁO CÁO VÀO FOLDER CỦA TỪNG NGÂN HÀNG ===============

if report_scores:
    print("\n---- XUẤT OUTPUT BÁO CÁO THEO NGÂN HÀNG ----")
    for report_data in report_scores:
        # Giả định bạn đặt tên file Báo cáo có chứa tên ngân hàng ở đầu. 
        # (Ví dụ: đặt tên file là "BIDV_baocao.pdf" hoặc "BIDV.pdf" -> `bank_name` sẽ tự trích xuất là "BIDV")
        pdf_filename = report_data["report_name"]
        
        # Đồng bộ folder output với biến TARGET_BANK cài đặt ở ô ban đầu
        bank_name = TARGET_BANK if TARGET_BANK != "*" else re.split(r'[_ -]', pdf_filename)[0].replace(".pdf", "")
        
        # SỬA LỖI ĐƯỜNG DẪN: DATA_DIR vốn dĩ đã trỏ vào data/<TARGET_YEAR>
        # Cấu trúc mục tiêu: data/<TARGET_YEAR>/<TARGET_BANK>/scoring
        scoring_output_dir = DATA_DIR / bank_name / "scoring"
        scoring_output_dir.mkdir(parents=True, exist_ok=True)
        
        report_output_path = scoring_output_dir / f"scored_{pdf_filename.replace('.pdf', '')}.json"
        
        with open(report_output_path, 'w', encoding='utf-8') as out_f:
            json.dump({"reports": [report_data]}, out_f, ensure_ascii=False, indent=4)
            
        print(f"✓ Đã lưu file điểm báo cáo của [{bank_name}] tại: {report_output_path}")

Đã tải thành công file báo cáo: Vietcombank_2023_ESG.pdf

Đang xử lý báo cáo: Vietcombank_2023_ESG.pdf
  -> Đã trích xuất và lưu nội dung text thô vào: Vietcombank_2023_ESG_raw.txt
  -> Đang tiền xử lý (Word Segmentation) văn bản...
  -> Đang tính toán điểm Total Score và Comparative Score...

---- XUẤT OUTPUT BÁO CÁO THEO NGÂN HÀNG ----
✓ Đã lưu file điểm báo cáo của [Vietcombank] tại: d:\NCKH\Thread_1\data\2023\Vietcombank\scoring\scored_Vietcombank_2023_ESG.json
